# 📊 PPLTool - Piped Processing Language Query Generation

```mermaid
%%{init: {'theme':'base', 'themeVariables': { 'primaryColor':'#16A085', 'primaryTextColor':'#fff', 'primaryBorderColor':'#138D75', 'lineColor':'#F39C12', 'secondaryColor':'#3498DB', 'tertiaryColor':'#27AE60', 'fontSize':'16px'}}}%%
graph TB
    A[👤 Natural Language<br/>Show error logs] --> B[🤖 Flow Agent]
    B --> C{📊 PPLTool}
    C --> D[🎯 LLM Model]
    D --> E[📝 Generate PPL Query]
    E --> F[✅ Valid PPL]
    F --> G[🔍 Execute Query]
    G --> H[📊 Results]
    
    style A fill:#3498DB,stroke:#2980B9,color:#fff
    style C fill:#16A085,stroke:#138D75,color:#fff
    style D fill:#9B59B6,stroke:#8E44AD,color:#fff
    style E fill:#E67E22,stroke:#D35400,color:#fff
    style H fill:#27AE60,stroke:#229954,color:#fff
```

## 📚 Learning Objectives

1. ✅ Convert natural language to **PPL (Piped Processing Language)** queries
2. ✅ Use **PPL syntax** for log and event analysis
3. ✅ Build **analytics pipelines** with pipes
4. ✅ **Execute PPL queries** automatically
5. ✅ Simplify **complex log analysis** workflows

---

## 🎯 What is PPLTool?

**PPLTool** generates **Piped Processing Language (PPL)** queries from natural language. PPL is ideal for:
- 📊 **Log Analysis**: Filter, aggregate, transform log data
- 🔄 **Data Pipelines**: Chain operations with pipes
- 📈 **Analytics**: Time-series analysis, statistics
- 🎯 **Simplicity**: More intuitive than DSL for many use cases

**PPL Example**:
```sql
source=logs | where level='ERROR' | stats count() by service
```

---

## Step 1: Import Libraries

In [ ]:
import sys
import json
from datetime import datetime, timedelta

sys.path.append('..')
from agent_helpers import (
    get_os_client,
    configure_cluster_for_openai,
    create_openai_connector,
    register_and_deploy_openai_model,
    create_flow_agent,
    execute_agent,
    cleanup_resources
)

print("✅ Libraries imported!")

## Step 2: Setup Client and OpenAI

In [ ]:
client = get_os_client()
configure_cluster_for_openai(client)
connector_id = create_openai_connector(client)
model_id = register_and_deploy_openai_model(client, connector_id)
print(f"✅ Ready: {model_id}")

## Step 3: Create Sample Logs Index

In [ ]:
index_name = "application_logs"

if client.indices.exists(index=index_name):
    client.indices.delete(index=index_name)

client.indices.create(index=index_name)

# Sample log data
logs = [
    {"timestamp": "2025-11-09T10:00:00Z", "level": "INFO", "service": "api", "message": "Request processed"},
    {"timestamp": "2025-11-09T10:01:00Z", "level": "ERROR", "service": "api", "message": "Connection timeout"},
    {"timestamp": "2025-11-09T10:02:00Z", "level": "ERROR", "service": "database", "message": "Query failed"},
    {"timestamp": "2025-11-09T10:03:00Z", "level": "WARN", "service": "api", "message": "Slow response"},
    {"timestamp": "2025-11-09T10:04:00Z", "level": "INFO", "service": "worker", "message": "Job completed"},
    {"timestamp": "2025-11-09T10:05:00Z", "level": "ERROR", "service": "api", "message": "Authentication failed"},
]

for log in logs:
    client.index(index=index_name, body=log, refresh=True)

print(f"✅ Created {len(logs)} log entries")

## Step 4: Create Agent with PPLTool

In [ ]:
tools = [{
    "type": "PPLTool",
    "parameters": {
        "model_id": model_id,
        "model_type": "OPENAI",
        "execute": True,
        "input": '{"index": "${parameters.index}", "question": "${parameters.question}"}'
    }
}]

agent_id = create_flow_agent(
    client, "PPL_Query_Agent",
    "Generates and executes PPL queries from natural language",
    tools
)
print(f"✅ PPL agent created: {agent_id}")

## Step 5: Test Case 1 - Simple Error Log Query

In [ ]:
parameters = {
    "index": index_name,
    "question": "Show me all ERROR level logs"
}

print("❓ Question: Show me all ERROR level logs")
print("="*60)
response = execute_agent(client, agent_id, parameters)
print("\n📊 Generated PPL & Results:")
print(json.dumps(response, indent=2))

## Step 6: Test Case 2 - Count Errors by Service

In [ ]:
parameters = {
    "index": index_name,
    "question": "Count the number of errors grouped by service"
}

print("❓ Question: Count errors grouped by service")
print("="*60)
response = execute_agent(client, agent_id, parameters)
print("\n📊 PPL Query & Results:")
print(json.dumps(response, indent=2))

## Step 7: Test Case 3 - Filter by Service and Level

In [ ]:
parameters = {
    "index": index_name,
    "question": "Show API service logs with WARNING or ERROR level"
}

print("❓ Question: Show API service WARN/ERROR logs")
print("="*60)
response = execute_agent(client, agent_id, parameters)
print("\n📊 Results:")
print(json.dumps(response, indent=2))

## Step 8: Test Case 4 - Recent Errors

In [ ]:
parameters = {
    "index": index_name,
    "question": "What were the most recent 5 error messages?"
}

print("❓ Question: Most recent 5 error messages")
print("="*60)
response = execute_agent(client, agent_id, parameters)
print("\n📊 Results:")
print(json.dumps(response, indent=2))

## 🎓 Key Takeaways

### What We Learned:

1. **PPLTool Capabilities**:
   - ✅ Natural language → PPL query conversion
   - ✅ Automatic query execution
   - ✅ Log analysis and filtering
   - ✅ Aggregations and statistics

2. **PPL Syntax Examples**:
   ```sql
   # Simple filter
   source=logs | where level='ERROR'
   
   # Aggregation
   source=logs | stats count() by service
   
   # Multiple conditions
   source=logs | where level='ERROR' AND service='api'
   
   # Sorting and limiting
   source=logs | sort timestamp desc | head 10
   ```

3. **PPL vs DSL**:
   | Aspect | PPL | DSL |
   |--------|-----|-----|
   | Syntax | SQL-like | JSON |
   | Readability | High | Medium |
   | Use Case | Logs, Analytics | General Search |
   | Learning Curve | Gentle | Steep |

4. **Common PPL Commands**:
   - `source`: Specify index
   - `where`: Filter rows
   - `stats`: Aggregate data
   - `sort`: Order results
   - `head`: Limit results
   - `fields`: Select columns
   - `dedup`: Remove duplicates

5. **Configuration**:
   ```python
   {
       "type": "PPLTool",
       "parameters": {
           "model_id": model_id,
           "model_type": "OPENAI",
           "execute": True,  # Auto-execute query
           "input": '{"index": "...", "question": "..."}'
       }
   }
   ```

### Use Cases:

- 📊 **Log Analysis**: Find errors, warnings, patterns
- 🔍 **Troubleshooting**: Debug application issues
- 📈 **Monitoring**: Track service health
- 🎯 **Security**: Analyze access logs, detect anomalies

### Best Practices:

- ✅ **Time Ranges**: Always specify time bounds for large datasets
- ✅ **Index Patterns**: Use index patterns for time-series data
- ✅ **Field Names**: Ensure fields exist in the index
- ✅ **Result Limits**: Use `head` to limit result size

---

## 🧹 Cleanup

In [ ]:
# # cleanup_resources(
# #     client=client,
# #     agent_ids=[agent_id],
# #     model_ids=[model_id],
# #     connector_ids=[connector_id]
# # )
# # client.indices.delete(index=index_name)
# # print("✅ Cleanup complete!")

## 🚀 Next Steps

- **LogPatternTool**: Extract patterns from logs
- **LogPatternAnalysisTool**: Advanced log analysis
- **QueryPlanningTool**: DSL query generation

📚 [PPLTool Documentation](https://opensearch.org/docs/latest/ml-commons-plugin/agents-tools/tools/ppl-tool/)